### Latin square Simulation

- Latin square is the square with all rows and columns has distinct symbol: https://en.wikipedia.org/wiki/Latin_square
- In this notebook, the author will simulate the latent square for reasonable dimensions <= 11 and provide an est for total number of latent squares using SMC
- Additionally, the same idea will be applied to solve a sudoku puzzle. 

Let's start with some setup! Define:  
- $\pi_0$ is the uniform distribution of all squares n x n with each row is already a permutation from $1 - n$
so that we only need to consider the validity of columns.

- $\pi_1$ is the uniform distribution of all Latin squares n x n.

We will start from $\pi_0$ to generate samples from $\pi_T$

- Let's say if we directly apply the ideas of SMCS by building a list of intermediate distributions $\pi_0^{(1 - \lambda)} * \pi_1^{\lambda}$, what happen is that for all lambdas > 0, all intermediate dists will be collapsed to the target distribution as its pdf is exactly the same aft normalizing (i.e. = 0 if the square is non-Latin and = 1 otherwise)

The idea is interesting as we will try to not directly sample from our target dist but from an "approximate" version of it. 
Let's consider: 

$target = e^{-lambda * score(X)}$ with lambda is big enough value let say 1000. 

- Score(X) here is a score function to measure how "bad" a sample from a Latin properties.
- When X is Latin square, score(X) will be zero and target = 1, otherwise, this value is very small, thanks to the big lambda. By doing SMC on this alternative dist, most of the samples we received will expect to be Latin.

In [1]:
from libs import MCMC, SMC, AcceptanceTracker, test_samples

First of all, define a scorer class to maintain all scorer methods we will have 

In [2]:
from jax import numpy as jnp

import numpy as np 

class Scorer:
    @staticmethod
    def score_repetitive_columns(flat_board: jnp.array):
        n = np.sqrt(len(flat_board)).astype(int)
        assert len(flat_board) == n * n
        
        score = 0 
        for c in range(n):
            # symbols are 1..n, so the mask needs n + 1 slots
            seen = jnp.zeros(n + 1, dtype=bool)
            for x in flat_board[c::n]:
                score += jnp.where(seen[x], 1, 0)
                seen = seen.at[x].set(True)
        return score 

    @staticmethod
    def score_sudoku(flat_board: np.array):
        #TODO: implement
        return 0 

assert Scorer.score_repetitive_columns(jnp.array([1,2,1,2])) == 2
assert Scorer.score_repetitive_columns(jnp.array([1,2,2,1])) == 0

#### Latin Square Solver

Secondly, define the prior/target dist logpdfs we will sample from as well as the proposed_fn to move to the next state


In [3]:
import jax
import jax.numpy as jnp
from jax import random

import numpy as np 

def prior_dist_logpdf(flat_board: jnp.array):
    """
    Supports:
        flat_board.shape == (N,)
        flat_board.shape == (B, N)
    """
    board_size = flat_board.shape[-1]
    n = int(np.sqrt(board_size))

    log_prob = -n * jnp.log(jnp.arange(1, n + 1)).sum()

    if flat_board.ndim == 1:
        return log_prob
    else:
        return jnp.full((flat_board.shape[0],), log_prob)

def target_dist_logpdf(flat_board: jnp.array, score_fn, lam=1000):
    """
    Supports:
        (N,)  -> scalar
        (B,N) -> (B,)
    """
    if flat_board.ndim == 1:
        return -lam * score_fn(flat_board)
    else:
        scores = jax.vmap(score_fn)(flat_board)
        return -lam * scores

In [4]:
import jax
import jax.numpy as jnp
from jax import random
import numpy as np

def proposed_fn(flat_board: jnp.array, key):
    """
    Supports:
        (N,)
        (B,N)
    """

    def propose_one(board: np.array, key):
        board_size = board.shape[0]
        n = jnp.sqrt(board_size).astype(int)

        key1, key2 = random.split(key)

        row = random.randint(key1, (), 0, n)
        col = random.randint(key2, (), 0, n - 1)

        i = row * n + col
        j = i + 1

        board_i, board_j = board[i], board[j]
        board = board.at[i].set(board_j)
        board = board.at[j].set(board_i)
        return board 

    if flat_board.ndim == 1:
        return propose_one(flat_board, key)

    else:
        keys = random.split(key, flat_board.shape[0])
        return jax.vmap(propose_one)(flat_board, keys)

In [5]:
from jax import random, vmap
from jax import numpy as jnp 
from functools import partial

class LatinSquareSampler: 
    def __init__(self, n: int, score_fn):
        self.__score_fn = score_fn
        self.__key = random.key(100)
        self.__n = n  
        self.smc = SMC(
            dims = n * n, 
            target_dist_logpdf = partial(
                target_dist_logpdf, 
                score_fn = Scorer.score_repetitive_columns
            ), 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn, 
            key = self._split_key()[0]
        )
        # prior_dist_logpdf is already normalised over the (n!)^n row-permuted
        # squares, so Z_0 = 1 and the SMC increments alone estimate log(L_n).
        self.log_z = 0.0

    def _split_key(self, n = 2):
        split_keys = random.split(self.__key, n)
        self.__key = split_keys[0]
        return split_keys[1:]

    def __generate_initial_state(self, key):
        sub_keys = random.split(key, num = self.__n)
        perm = vmap(
            lambda key: random.permutation(key, jnp.arange(1, self.__n + 1)) 
        )(sub_keys)
        return perm.reshape(-1)

    def __generate_initial_states(self, no_samples: int):
        sub_keys = self._split_key(no_samples + 1)
        states = vmap(
            lambda key: self.__generate_initial_state(key)
        )(sub_keys)
        return states
    
    def run_smc(self, no_samples: int):
        samples = self.__generate_initial_states(no_samples)
        assert samples.shape == (no_samples, self.__n * self.__n)
        
        self.smc.reset(samples = samples)
        lam_list, diff_log_z = self.smc.build_intermediate_dists(max_steps=64, n_bisect=30, mcmc_iters=50)
        self.log_z += diff_log_z
        return lam_list

    def get_current_sample_list(self):
        return self.smc.get_current_sample_list()

    def est_total(self):
        return jnp.exp(self.log_z)  

Now let's test our solver!

In [6]:
from jax import numpy as jnp

grouth_truth = [
    1, 
    2, 
    12, 
    576, 
    161280, 
    812851200, 
    61479419904000, 
    108776032459082956800, 
    5524751496156892842531225600, 
    9982437658213039871725064756920320000, 
    776966836171770144107444346734230682311065600000
]

In [7]:
NDIMS = 5
latin_solver = LatinSquareSampler(
    n = NDIMS, 
    score_fn = Scorer.score_repetitive_columns
)

print(f"before log_z: {latin_solver.log_z}")

lam_list = latin_solver.run_smc(
    no_samples = 300_000
)

print(f"aft log_z: {latin_solver.log_z}")

samples = latin_solver.get_current_sample_list()

before log_z: 0.0
Current loop value: 0.0
Current loop value: 0.0003114324063062668
Current loop value: 0.0006248290883377194
Current loop value: 0.0009413041407242417
Current loop value: 0.001257865922525525
Current loop value: 0.0015730271115899086
Current loop value: 0.0018907072953879833
Current loop value: 0.002232622355222702
Current loop value: 0.0026592942886054516
aft log_z: 11.992090225219727


Display some Latin square samples 

In [8]:
for sample in samples[:10]:
    print(sample.reshape(NDIMS, NDIMS), "\n")

[[4 3 5 1 2]
 [5 1 3 2 4]
 [1 4 2 3 5]
 [2 5 1 4 3]
 [3 2 4 5 1]] 

[[3 5 2 1 4]
 [5 2 1 4 3]
 [1 4 3 2 5]
 [4 1 5 3 2]
 [2 3 4 5 1]] 

[[1 5 3 4 2]
 [3 4 5 2 1]
 [2 3 4 1 5]
 [5 1 2 3 4]
 [4 2 1 5 3]] 

[[3 1 5 2 4]
 [4 2 3 1 5]
 [5 3 2 4 1]
 [2 4 1 5 3]
 [1 5 4 3 2]] 

[[4 5 2 1 3]
 [3 4 1 5 2]
 [2 1 3 4 5]
 [1 3 5 2 4]
 [5 2 4 3 1]] 

[[4 1 3 2 5]
 [5 2 1 4 3]
 [2 4 5 3 1]
 [1 3 2 5 4]
 [3 5 4 1 2]] 

[[3 1 4 2 5]
 [4 2 1 5 3]
 [2 3 5 4 1]
 [5 4 3 1 2]
 [1 5 2 3 4]] 

[[4 3 2 5 1]
 [5 4 1 3 2]
 [2 1 5 4 3]
 [1 5 3 2 4]
 [3 2 4 1 5]] 

[[3 4 5 1 2]
 [4 1 3 2 5]
 [1 5 2 4 3]
 [2 3 4 5 1]
 [5 2 1 3 4]] 

[[5 2 3 1 4]
 [3 4 1 2 5]
 [2 5 4 3 1]
 [1 3 5 4 2]
 [4 1 2 5 3]] 



In [9]:
# for sample in samples[:1000]:
#     assert Scorer.score_repetitive_columns(sample) == 0

In [10]:
latin_solver.est_total()

Array(161472.52, dtype=float32)

In [35]:
from jax import vmap, jit
from jax import numpy as jnp
import numpy as np

def solve(dims: int):
    latin_solver = LatinSquareSampler(n = dims, score_fn = Scorer.score_repetitive_columns)
    latin_solver.run_smc(no_samples = 100_000)
    return latin_solver.est_total() 

ans = [solve(dims) for dims in range(1, 12)]

Current loop value: 0.0
Current loop value: 0.0
Current loop value: 0.00046207010746002197
Current loop value: 0.0009228394483216107
Current loop value: 0.0014409529976546764
Current loop value: 0.0
Current loop value: 0.000373045913875103
Current loop value: 0.0007411172846332192
Current loop value: 0.0011003254912793636
Current loop value: 0.0014632996171712875
Current loop value: 0.0018735595513135195
Current loop value: 0.0
Current loop value: 0.000311901792883873
Current loop value: 0.0006266698474064469
Current loop value: 0.0009419576963409781
Current loop value: 0.0012581304181367159
Current loop value: 0.0015727865975350142
Current loop value: 0.0018906649202108383
Current loop value: 0.00223164027556777
Current loop value: 0.002657831646502018
Current loop value: 0.0
Current loop value: 0.0002643689513206482
Current loop value: 0.0005306614330038428
Current loop value: 0.000800865120254457
Current loop value: 0.001073599560186267
Current loop value: 0.0013490261044353247
Curr

In [37]:
for dims in range(1, 12):
    print(f"est value = {ans[dims - 1]}, grouth_truth = {grouth_truth[dims - 1]}")

est value = 1.0, grouth_truth = 1
est value = 1.9999990463256836, grouth_truth = 2
est value = 12.091897010803223, grouth_truth = 12
est value = 576.5264892578125, grouth_truth = 576
est value = 163895.25, grouth_truth = 161280
est value = 825391424.0, grouth_truth = 812851200
est value = 53928850882560.0, grouth_truth = 61479419904000
est value = 8.70280078000516e+19, grouth_truth = 108776032459082956800
est value = 1.6330388971100054e+27, grouth_truth = 5524751496156892842531225600
est value = 2.950920751957928e+34, grouth_truth = 9982437658213039871725064756920320000
est value = inf, grouth_truth = 776966836171770144107444346734230682311065600000


#### Sudoku Solver

In [ ]:
from jax import random
from functools import partial

class SudokuSolver: 
    def __init__(self, n: int, configuration: jnp.array, score_fn):
        self.__score_fn = score_fn
        self.smc = SMC(
            dims = n * n, 
            target_dist_logpdf = target_dist_logpdf, 
            prior_dist_logpdf = prior_dist_logpdf, 
            proposed_fn = proposed_fn, 
            key = random.key(0)
        )
        self.z = 0 
        self.configuration = configuration

    def run_smc(self):
        r"""
            build intermediate + samples from the valid sudoku   
        """
        ... 

    def solve(self):
        r"""
            iteratively take from smc sample until has something with score = 0 
        """
        ... 